In [2]:
import torch
import numpy as np
from ase.io import read
from mace.calculators import mace_mp, mace_off

/home/grethel/env/mace/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


In [ ]:
path = "/home/grethel/dev/quests/examples/gap20/Graphite.xyz"
frames = read(path, index=":")

model = "small"
# model = "medium"
# model = "large"

model_type = "mp"
# model_type = "off"

# --------------------
# Load MACE model (once)
# --------------------
if model_type == "off":
    calc = mace_off(
        model=model,
        dispersion=False,
        default_dtype="float32",
        device="cuda:0",
    )
else:
    calc = mace_mp(
        model=model,
        dispersion=False,
        default_dtype="float32",
        device="cuda:0",
    )

model = calc.models[0]
model.eval()

# --------------------
# Hook to grab embeddings
# --------------------
embeddings = {}

def hook_fn(module, inputs, output):
    embeddings["emb"] = output.clone().detach().cpu()

hook = model.products[-1].register_forward_hook(hook_fn)

energies = []
all_embeddings = []

torch.set_grad_enabled(False)

# --------------------
# Loop over frames
# --------------------
for atoms in frames:
    # Build batch directly (NO ASE energy call)
    batch = calc._atoms_to_batch(atoms)
    data = batch.to_dict()

    # Move tensors to GPU
    for k, v in data.items():
        if torch.is_tensor(v):
            if v.is_floating_point():
                data[k] = v.to("cuda:0", dtype=torch.float32)
            else:
                data[k] = v.to("cuda:0")

    # Forward pass WITHOUT forces / stress
    out = model(
        data,
        training=False,
        compute_force=False,
        compute_virials=False,
        compute_stress=False,
        compute_displacement=False,
    )

    energies.append(out["energy"].item())
    all_embeddings.append(embeddings["emb"])

hook.remove()

# --------------------
# Stack results
# --------------------
embeddings_all = torch.vstack(all_embeddings).numpy()
energies = np.array(energies)

print("embeddings_all:", embeddings_all.shape, embeddings_all.dtype)
print("energies:", energies.shape, energies.dtype)


The model is distributed under the Academic Software License (ASL) license, see https://github.com/gabor1/ASL 
 To use the model you accept the terms of the license.
ASL is based on the Gnu Public License, but does not permit commercial use
Cached MACE model to /home/grethel/.cache/mace/MACE-OFF23_large.model
Using MACE-OFF23 MODEL for MACECalculator with /home/grethel/.cache/mace/MACE-OFF23_large.model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.


/home/grethel/env/mace/lib/python3.11/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Using head Default out of ['Default']
Default dtype float32 does not match model dtype float64, converting models to float32.
embeddings_all: (12220, 224) float32
energies: (160,) float64


### Alternative method

In [ ]:
path = "/home/grethel/dev/quests/examples/gap20/Graphite.xyz"
frames = read(path, index=":")[135:136]

energy_lst = []
embeddings_lst = []

calc = mace_mp(
    model="medium",
    dispersion=False,
    default_dtype="float32",
    device="cuda:0",
)

torch.set_grad_enabled(False)
model = calc.models[0]

for atoms in frames:
    atoms.calc = calc
    embeddings = {}

    def hook_fn(module, inputs, output):
        embeddings["embeddings"] = output.clone().cpu()

    hook_handle = model.products[-1].register_forward_hook(hook_fn)

    energy = atoms.get_potential_energy()

    hook_handle.remove()

    energy_lst.append(energy)
    embeddings_lst.append(embeddings["embeddings"])

embeddings_all = torch.vstack(embeddings_lst).detach().numpy()

In [9]:
print("Energy:", energy)
print("Embedding shape:", embeddings_all.shape)
# print("Embeddings:", embeddings["embeddings"])

Energy: -903.654052734375
Embedding shape: (100, 128)


### Convert Equivariant Embeddings to Invariant

In [49]:
model = "small"
# model = "medium"
# model = "large"

# model_type = "mp"
model_type = "off"

# --------------------
# Load MACE model (once)
# --------------------
if model_type == "off":
    calc = mace_off(
        model=model,
        dispersion=False,
        default_dtype="float32",
        device="cuda:0",
    )
else:
    calc = mace_mp(
        model=model,
        dispersion=False,
        default_dtype="float32",
        device="cuda:0",
    )
# 128x0e+128x1o+128x2e+128x3o
model = calc.models[0]
model
print(model.interactions[-1].linear.irreps_out)

Using MACE-OFF23 MODEL for MACECalculator with /home/grethel/.cache/mace/MACE-OFF23_small.model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.
Using head Default out of ['Default']
Default dtype float32 does not match model dtype float64, converting models to float32.
96x0e+96x1o+96x2e+96x3o


/home/grethel/env/mace/lib/python3.11/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


In [41]:
import numpy as np
import torch

# --- Load embeddings ---

orig = np.load("/home/grethel/dev/quests/embeddings/mace_mp_small_Graphene_one_frame.npz", allow_pickle=True)
rot  = np.load("/home/grethel/dev/quests/embeddings/mace_mp_small_Graphene_one_frame_rotated.npz", allow_pickle=True)

emb_orig = torch.from_numpy(orig["embeddings"])  # e.g. shape (200, 128, 16)
emb_rot  = torch.from_numpy(rot["embeddings"])   # same shape

print("Original embedding shape:", emb_orig.shape)
print("Rotated  embedding shape:", emb_rot.shape)

# --- Helper to compute invariant features ---

def compute_invariant_features(x):
    """
    x: (n_atoms, channels, 16)
    We interpret last dim as irreps:
      [0]         -> l=0 scalar
      [1:4]       -> l=1 vector (3 comps)
      [4:9]       -> l=2 tensor (5 comps)
      [9:16]      -> l=3 tensor (7 comps)
    Returns invariant descriptor: (n_atoms, channels * 4)
    """
    # scalar part (already invariant)
    inv_l0 = x[:, :, 0]            # shape (n_atoms, channels)

    # compute norms of each block
    inv_l1 = torch.norm(x[:, :, 1:4], p=2, dim=-1)   # (n_atoms, channels)
    inv_l2 = torch.norm(x[:, :, 4:9], p=2, dim=-1)   # (n_atoms, channels)
    inv_l3 = torch.norm(x[:, :, 9:16], p=2, dim=-1)  # (n_atoms, channels)

    # concatenate all invariants
    return torch.cat([inv_l0, inv_l1, inv_l2, inv_l3], dim=-1)  # (n_atoms, channels*4)


# --- Compute invariant descriptors ---

inv_orig = compute_invariant_features(emb_orig)   # (n_atoms, 128*4)
inv_rot  = compute_invariant_features(emb_rot)    # (n_atoms, 128*4)

print("Invariant shape:", inv_orig.shape)

# --- Compare original vs rotated invariants ---

# difference per element
diff = torch.abs(inv_orig - inv_rot)

max_diff = diff.max().item()
mean_diff = diff.mean().item()

print(f"Max absolute diff (invariants): {max_diff:.3e}")
print(f"Mean absolute diff (invariants): {mean_diff:.3e}")

if max_diff < 1e-6:
    print("✅ Invariant features are numerically equal (rotation invariance confirmed).")
else:
    print("⚠️ Invariant features differ by more than expected numeric noise!")

# --- Optionally show that raw equivariant parts differ ---

# you can flatten if needed
eqv_orig_flat = emb_orig.reshape(-1)
eqv_rot_flat  = emb_rot.reshape(-1)

eqv_diff = torch.abs(eqv_orig_flat - eqv_rot_flat)
print(f"Max equivariant diff (raw): {eqv_diff.max().item():.3e}")


Original embedding shape: torch.Size([200, 128, 16])
Rotated  embedding shape: torch.Size([200, 128, 16])
Invariant shape: torch.Size([200, 512])
Max absolute diff (invariants): 2.615e-06
Mean absolute diff (invariants): 6.189e-08
⚠️ Invariant features differ by more than expected numeric noise!
Max equivariant diff (raw): 2.615e-06


In [ ]:
def mace_to_invariant(emb: torch.Tensor) -> torch.Tensor:
    """
    Convert a MACE interaction embedding to rotation-invariant form.

    Args:
        emb: Tensor of shape (N, channels, 16)
             where 16 = 1 + 3 + 5 + 7 for l=0,1,2,3.

    Returns:
        inv: Tensor of shape (N, channels * 4)
             containing invariant features per atom.
    """
    # check input shape
    if emb.ndim != 3 or emb.shape[-1] != 16:
        raise ValueError("Expected emb with shape (N, channels, 16)")

    # l=0 scalars are already invariant
    inv_l0 = emb[:, :, 0]  # (N, channels)

    # compute norms of each equivariant block
    inv_l1 = torch.norm(emb[:, :, 1:4], dim=-1)   # (N, channels)
    inv_l2 = torch.norm(emb[:, :, 4:9], dim=-1)   # (N, channels)
    inv_l3 = torch.norm(emb[:, :, 9:16], dim=-1)  # (N, channels)

    # concatenate into final invariant descriptor
    inv = torch.cat([inv_l0, inv_l1, inv_l2, inv_l3], dim=-1)  # (N, channels*4)
    return inv


data = np.load("/home/grethel/dev/quests/embeddings/mace_mp_small_Graphene_one_frame.npz", allow_pickle=True)
data_emb = torch.from_numpy(orig["embeddings"])  # e.g. shape (200, 128, 16)
data_inv = mace_to_invariant(data_emb)  # shape (200, 512)
print("Invariant shape:", data_inv.shape)

Invariant shape: torch.Size([200, 512])


### Float64 Demo

In [7]:
import torch
torch.set_default_dtype(torch.float64)
import numpy as np
from ase.build import bulk
from mace.calculators import mace_mp

device = "cuda"
# device = "cpu"
dtype = torch.float64

atoms = bulk("Al", "fcc", a=4.05, cubic=True) * (2, 2, 2)

# model_size="small"
# model_size="medium"
model_size = "large"

calc = mace_mp(
    model=model_size,
    device=device,
    default_dtype="float64", 
    enable_cueq=True,
)

atoms.calc = calc

energy = atoms.get_potential_energy()

print(energy)

Using Materials Project MACE for MACECalculator with /home/grethel/.cache/mace/MACE_MPtrj_20229model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


/home/grethel/env/mace/lib/python3.11/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/grethel/env/mace/lib/python3.11/site-packages/torch/jit/_serialization.py:176: DeprecationWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/grethel/env/mace/lib/python3.11/site-packages/torch/jit/_serialization.py:176: DeprecationWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/grethel/env/mace/lib/python3.11/site-packages/torch/jit/_serialization.py:176: DeprecationWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/grethel/env/mace/lib/python3.11/site-packages/mace/modules/models.py:84: UserWarning: To copy construct from a tensor, it 

Using head Default out of ['Default']
Converting models to CuEq for acceleration


/home/grethel/env/mace/lib/python3.11/site-packages/cuequivariance_torch/operations/tp_fully_connected.py:142: DeprecationWarning: `use_fallback` is deprecated, please use `method` instead
  warnings.warn(
/home/grethel/env/mace/lib/python3.11/site-packages/cuequivariance_torch/operations/linear.py:125: DeprecationWarning: `use_fallback` is deprecated, please use `method` instead
  warnings.warn(
/home/grethel/env/mace/lib/python3.11/site-packages/cuequivariance_torch/operations/linear.py:125: DeprecationWarning: `use_fallback` is deprecated, please use `method` instead
  warnings.warn(
/home/grethel/env/mace/lib/python3.11/site-packages/cuequivariance_torch/operations/linear.py:125: DeprecationWarning: `use_fallback` is deprecated, please use `method` instead
  warnings.warn(
/home/grethel/env/mace/lib/python3.11/site-packages/cuequivariance_torch/primitives/segmented_polynomial.py:155: UserWarning: Hello! It looks like you're using code that was written for an older version of this l

RuntimeError: CUDA error: CUBLAS_STATUS_NOT_INITIALIZED when calling `cublasDgemm( handle, opa, opb, m, n, k, &alpha, a, lda, b, ldb, &beta, c, ldc)`